# 02 — LightGBM 단일 회귀

Evidence-driven Wide HPO + anchor enqueue + 후처리 매트릭스.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/02_reg_single/lgbm/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` ([strategy_common.md §1](../strategy_common.md))
- **HPO**: 150 trial, anchor 첫 trial enqueue ([strategy.md §5.1](strategy.md), 출처: [4_output_이전자료/final/reg_only/lgbm/best_params.json](../../4_output_이전자료/final/reg_only/lgbm/best_params.json)), wide range ([§7.1](strategy.md))
- **손실함수**: `regression / poisson / tweedie` 3종 ([§6](strategy.md))
- **target transform**: `'none'` 고정 (strategy_common §24 — log1p_check 검증)

## 모듈 의존성 ([strategy.md §13](strategy.md))

이 노트북은 아래 모듈 변경을 전제로 한다 (다른 세션 작업):

1. `3_modeling_이전자료/final/modules/` → `3_modeling/modules/` 이관
2. `models.py` — `lgbm_space`의 `tweedie_*` categorical → `tweedie` + `tweedie_variance_power=suggest_float(1.05, 1.95)`
3. `hpo.py` — `run_hpo(..., enqueue_trials=[anchor])` 인자 추가 + trial 내부 target_transform 분기 (tweedie 시 OFF)
4. `postprocess.py` — `agg_methods`에 `Q25`, `Q75` 추가 + `zero_clip_space='log'` 분기

## 1. 환경 설정 + 모듈 import

In [1]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip = cleaning/outlier/scaling 등
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 기존 실험 산출물 (RESUME용)
RESUME                 = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../setup.py만
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
        # 모듈 경로 등록 — Colab 런타임은 세션마다 sys.path가 초기화됨
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 공통 유틸: 경로 상수(OUTPUT_DIR, DATA_DIR), 컬럼 상수(TARGET_COL, KEY_COL), SEED
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
# 전처리 모듈(2_preprocessing/)과 모델링 모듈(3_modeling/modules/) 모두 sys.path 등록
# 노트북 위치가 달라도 PROJECT_ROOT 기준 절대경로로 접근 → Colab·로컬 동일
PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

# preprocess.run: 전체 전처리 파이프라인(결측/이상치/스케일/집계)
# hpo: HPO(run_hpo) + 재학습(refit_best) + 산출물 저장(save_artifacts)
# models: 모델 레지스트리 (AVAILABLE_MODELS 리스트 + 모델 생성 팩토리)
from modules import preprocess, hpo, models   # noqa: E402  (preprocess.run, hpo.run_hpo/refit_best/save_artifacts, models 레지스트리)
# meta_features: position·die_xy 메타피처 생성 (run_wf_xy 파싱 기반)
from meta_features import add_meta_features

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

setup 완료


PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
Available models: ['lgbm', 'xgb', 'catboost', 'et', 'enet', 'zitboost']


## 2. 실험 설정

In [2]:
# 모델 고정 (이 노트북은 LGBM 단일)
MODEL_NAME = 'lgbm'
EXP_ID     = f'reg-{MODEL_NAME}-002'
EXP_MEMO   = 'Evidence-driven Wide + anchor enqueue (1차 OOF=0.005521)'
USER       = 'jh'

# Optuna 예산
N_TRIALS = 3000
N_FOLDS  = 5
# N_STARTUP_TRIALS: 랜덤 탐색 후 TPE 전환 — 초반 공간 커버리지 확보
N_STARTUP_TRIALS = 50

N_JOBS = -1   # 모델 학습 병렬도 (-1 = 전체 코어)
TIMEOUT_SEC      = 20 * 60 * 60  # 초 단위, None=무제한 (Colab 타임아웃 대비)

# TARGET_TRANSFORM='none': 트리는 log1p 등 target 변환이 RMSE에 유의미한 차이 없음
TARGET_TRANSFORM = 'none'  # 트리는 'none' 통일 (log1p와 사실상 동등)
CLIP_Y_EXTREME   = True

# 출력 경로 — EXP_ID 끝자리('002')를 하위 폴더명으로
OUT_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME, EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# 트리 공통 전처리 고정 파라미터 — study 시작 전 1회만 적용
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# anchor — 1차 best HP. trial 0으로 강제 enqueue해서 1차 성능을 잃지 않음
LGBM_ANCHOR = {
        # 손실 함수: Poisson deviance → 0 많은 비음수 분포(health)에 MSE보다 적합
    'objective':         'poisson',
    'n_estimators':      957,
    'learning_rate':     0.00602,
        # 구조 HP: num_leaves(≤ 2^max_depth), max_depth, min_child_samples(잎 최소 샘플 수)
    'num_leaves':        379,
    'max_depth':         10,
    'min_child_samples': 343,
        # 서브샘플링: 행(subsample) + 열(colsample_bytree) — 약신호 feature 다양성 확보
    'subsample':         0.711,
    'subsample_freq':    1,
    'colsample_bytree':  0.597,
        # 정규화: L1(reg_alpha) + L2(reg_lambda), min_split_gain, path_smooth(리프 평탄화)
    'reg_alpha':         0.00840,
    'reg_lambda':        0.000151,
    'min_split_gain':    1.016e-04,
    'path_smooth':       24.81,
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}, N_JOBS={N_JOBS}, TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')

EXP: reg-lgbm-002 | USER: jh
N_TRIALS=1, N_FOLDS=5, N_JOBS=7, TIMEOUT_SEC=None
TARGET_TRANSFORM=none | CLIP_Y_EXTREME=True
OUT_DIR=C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\02_reg_single\lgbm


## 3. 데이터 로드 + target clip + transform

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
# split_xs: xs['split'] 컬럼 기준으로 train/val/test 행 분리
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip — 학습 입력 안정화 (원본 ys는 보존)
# ys는 원본 보존 — clip/transform은 ys_input 복사본에만 적용 (누수 방지)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# 트리 모델은 target 변환이 결과를 거의 안 바꿔서 'none'으로 고정 (transform 함수 둘 다 None)
# 트리는 target 변환이 RMSE를 유의하게 낮추지 않음 (EDA max|r|=0.037 약신호)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM} (strategy_common §24 — 트리 target_transform=none 통일)')

[load_xs] all-NaN 행 407개 제거 → 174,573행


[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572


[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729


xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] none (strategy_common §24 — 트리 target_transform=none 통일)


## 4. 전처리 (PP_FIXED 고정 — strategy_common.md §1)

In [4]:
# PP_FIXED: missing_threshold(결측 제거 기준), corr_threshold(다중공선성), add_indicator(결측 indicator 추가)
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# 메타피처: 트리는 position을 raw 정수로, die_x/die_y를 연속형으로 추가
# position_mode='raw': 위치를 1~4 정수로 직접 피처 추가 (범주형 OHE 없음)
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

# fit은 train에서만, transform은 train/val/test 모두 — 데이터 누수 방지
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033


[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개


    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)



[고결측 제거] threshold=30%
  제거: 5개, 잔여: 923개


    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)



...


  [요약] 343,494 → 공간(161,870) → lot(100,428) → 전체(81,196) → 잔여(0)


[고상관 제거] threshold=0.96, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.96
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 631)

클리닝 완료: 1033 → 564 features (469개 제거)
  + indicator 컬럼: 9개 → 총 573개
  train: (104748, 631)
  val:   (34908, 631)
  test:  (34916, 631)
이상치 처리 파이프라인 시작 (method=winsorize)


[이상치 탐지] IQR × 1.5
  이상치 > 5%: 112개
  이상치 > 10%: 64개


[Winsorization] lower=0%, upper=99%
  적용 feature: 573개

이상치 처리 완료 (method=winsorize)
  train: (104748, 631)


[add_meta_features] position_mode='raw', use_die_xy=True → position=['position'], die_xy=['die_x', 'die_y'] (feat_cols: 576)

[전처리 완료] feat_cols: 576


## 5. Optuna HPO (anchor 첫 trial enqueue + wide range)

In [5]:
# study_meta: HPO 실험 메타정보를 DB(user_attr)에 박제 → trial별 EXP_ID/anchor/PP 설정 조회 가능
study_meta_for_save = {
    'exp_id':              EXP_ID,
    'exp_memo':            EXP_MEMO,
    'user':                USER,
    'model_name':          MODEL_NAME,
    'target_transform':    TARGET_TRANSFORM,
    'clip_y_extreme':      CLIP_Y_EXTREME,
        # effective_pp_params: preprocess.run이 실제로 사용한 파라미터 (FIXED 기본값 + override 반영)
    'effective_pp_params': pp['effective_params'],
    'n_trials':            N_TRIALS,
    'n_folds':             N_FOLDS,
    'n_jobs':              N_JOBS,
        # n_startup_trials: TPE 이전 랜덤 탐색 횟수 — 공간 초기 커버리지 확보
    'n_startup_trials':    N_STARTUP_TRIALS,
    'timeout_sec':         TIMEOUT_SEC,
    'seed_kfold':          SEED,
    'anchor':              LGBM_ANCHOR,
}

# anchor를 trial 0으로 강제하려면 study를 먼저 만들어 enqueue → 그 다음 run_hpo가 같은 study(이름·storage)에 이어서
# Optuna study 설정 — storage=SQLite로 trial 결과 영구 저장 (Colab 재시작 후 RESUME 가능)
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
# direction='minimize': OOF RMSE 최소화. sampler·pruner 설정은 study 생성 시 1회만
_study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=RESUME,
        # TPESampler: multivariate=True → 파라미터 간 상관 모델링. seed=None → 실행마다 다른 탐색 경로
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
        # MedianPruner: n_warmup=10 trial 이후 중간값 미달 trial 조기 중단 → 탐색 효율 향상
    pruner=MedianPruner(n_warmup_steps=10),
)
# enqueue: 기존 trial이 없을 때만 anchor를 trial 0으로 강제 → RESUME 시 중복 방지
hpo.enqueue_anchor(_study, LGBM_ANCHOR)   # 기존 trial 없을 때만 enqueue (RESUME 시 중복 방지)

# HPO 본체: unit 단위 KFold OOF → unit RMSE를 minimize. val/test RMSE는 매 trial user_attr에 기록
res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
        # n_jobs: 병렬 fold 학습 (-1=전체 코어). N_FOLDS=5이므로 최대 5 worker가 동시 학습
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
    timeout=TIMEOUT_SEC,
    user_attrs=study_meta_for_save,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
# res에서 best trial 정보 추출 → 재학습(다음 cell)에 사용
study                 = res['study']
best_params_for_refit = res['best_params']
# best_value: HPO 목적함수(OOF RMSE) 최솟값 → study_meta에 저장하여 산출물과 함께 기록
study_meta_for_save['hpo_best_value'] = float(res['best_value'])

# trial 0이 anchor 키들을 그대로 갖고 있는지 확인 (enqueue 정상 동작)
first_trial_params = study.trials[0].params
anchor_keys_present = {k: first_trial_params.get(k) for k in LGBM_ANCHOR if k in first_trial_params}
print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'[검증] trial 0 params (anchor 키만): {anchor_keys_present}')
print(f'best_params = {best_params_for_refit}')

[I 2026-05-09 15:41:11,024] A new study created in RDB with name: reg-lgbm-002


[enqueue] anchor 첫 trial로 강제 (13 HP)


[I 2026-05-09 15:41:11,680] Using an existing study with name 'reg-lgbm-002' instead of creating a new one.


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-05-09 15:45:27,121] Trial 0 finished with value: 0.005521579670792027 and parameters: {'n_estimators': 957, 'learning_rate': 0.00602, 'num_leaves': 379, 'max_depth': 10, 'min_child_samples': 343, 'subsample': 0.711, 'subsample_freq': 1, 'colsample_bytree': 0.597, 'reg_alpha': 0.0084, 'reg_lambda': 0.000151, 'min_split_gain': 0.0001016, 'path_smooth': 24.81, 'objective': 'poisson'}. Best is trial 0 with value: 0.005521579670792027.

[HPO 완료] best OOF RMSE = 0.005522
[검증] trial 0 params (anchor 키만): {'objective': 'poisson', 'n_estimators': 957, 'learning_rate': 0.00602, 'num_leaves': 379, 'max_depth': 10, 'min_child_samples': 343, 'subsample': 0.711, 'subsample_freq': 1, 'colsample_bytree': 0.597, 'reg_alpha': 0.0084, 'reg_lambda': 0.000151, 'min_split_gain': 0.0001016, 'path_smooth': 24.81}
best_params = {'n_estimators': 957, 'learning_rate': 0.00602, 'num_leaves': 379, 'max_depth': 10, 'min_child_samples': 343, 'subsample': 0.711, 'subsample_freq': 1, 'colsample_bytree': 0.597,

## 6. Best trial 재학습 (K-fold OOF)

In [6]:
# best HP로 5-fold 재학습 → die-level OOF / val / test 예측 (val·test는 fold 평균)
# refit: HPO best_params로 N_FOLDS번 재학습 → die·unit 레벨 OOF/val/test 예측 생성
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params_for_refit,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

# 후처리 이전(mean 집계) unit RMSE를 정답과 정렬해 계산 — train(OOF) / val / test
# die-level 예측을 unit-level mean 집계 후 RMSE 계산 (후처리 최적화 이전 baseline)
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))

# val·test는 fold 평균 예측 (각 fold 모델이 동일 val/test에 predict → 평균)
y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u       = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
val_rmse    = float(np.sqrt(np.mean((val_u.values - y_val_true.values)**2)))

y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u      = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse   = float(np.sqrt(np.mean((test_u.values - y_test_true.values)**2)))

# segment 분해: train y max=1.0 / val y max≈0.17 → RMSE 스케일 차이가 크므로 세트별 함께 확인
print(f'\n[Refit 완료] (original space, postprocess 이전)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

[refit fold 1/5] tr_units=20949, vl_units=5238


[refit fold 2/5] tr_units=20949, vl_units=5238


[refit fold 3/5] tr_units=20950, vl_units=5237


[refit fold 4/5] tr_units=20950, vl_units=5237


[refit fold 5/5] tr_units=20950, vl_units=5237

[Refit 완료] (original space, postprocess 이전)
  OOF  unit RMSE = 0.005522
  val  unit RMSE = 0.005734
  test unit RMSE = 0.008428


## 7. 후처리 매트릭스 + 산출물 저장

[strategy.md §10](strategy.md), [strategy_common.md §10·§12](../strategy_common.md):
- 집계 8종 (mean/median/max/min/trimmed_mean/weighted/Q25/Q75) — §13 #7 변경
- zero_clip log space 비교 — §13 #6 변경

In [7]:
# 후처리 설정 — die→unit 집계 8종 중 best + zero_clip 임계값 탐색. trees는 log space 안 씀(target_transform='none'), π threshold 없음
# 후처리: die→unit 집계 방식 8종 × zero_clip 임계값 그리드 탐색 → val RMSE 최소 조합 선택
POSTPROCESS_CONFIG = {
    'agg_methods':      ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
        # zero_clip: 임계값 이하 예측을 0으로 snap — False Positive(소량 예측) 제거
    'zero_clip_range':  (0.001, 0.015),
    'zero_clip_step':   0.001,
    'zero_clip_log_space': TARGET_TRANSFORM == 'log1p',
    'use_pi_threshold': False,
}

# fold_models.pkl + best_params.json + die/unit CSV 6개 저장 (postprocess_config 주면 unit CSV는 튜닝값)
# save_artifacts: fold_models.pkl + best_params.json + die·unit CSV 6개 → stacking 재사용 가능
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

# 저장된 파일 목록
# 저장 완료 후 파일 목록 출력 (파일명 + 크기) — 누락 파일 즉시 확인
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
# Colab 환경이면 OUT_DIR을 zip으로 묶어 로컬 PC에 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass

[I 2026-05-09 15:49:56,672] A new study created in memory with name: no-name-d03f9735-1e9d-4699-becf-dcd9dd3bfc48


[Position weights / Optuna 50t] best=0.005521, w=[0.043, 0.19, 0.265, 0.502]


[Aggregation] RMSEs: {'mean': 0.005522, 'median': 0.005522, 'max': 0.005526, 'min': 0.005526, 'trimmed_mean': 0.005522, 'weighted': 0.005521, 'Q25': 0.005522, 'Q75': 0.005523}
[Aggregation] best=weighted (0.005521)
[zero_clip] best=0.0010 (0.005521)
[Postprocess] best_agg=mean, pi_th=None, zero_clip=0.001, train_rmse=0.005521, val_rmse=0.005731
  baseline_mean                  val_rmse=0.005733588172913485
  after_agg(mean)                val_rmse=0.005733588172913485
  after_pi_th                    val_rmse=0.005733588172913485
  after_zero_clip                val_rmse=0.005731450903922894
  [decision] aggregation    weighted rejected (val 0.005734 <= 0.005736) -> keep mean
  [decision] zero_clip      0.0010 adopted (val 0.005734 -> 0.005731)


[save_artifacts] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\02_reg_single\lgbm 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=tuned)
  best_params.json                      11.5 KB
  fold_models.pkl                   44,189.3 KB
  oof_die.csv                        5,582.8 KB
  oof_unit.csv                         939.5 KB
  optuna_jh_reg-lgbm-002.db            112.0 KB
  test_die.csv                       1,860.8 KB
  test_unit.csv                        313.1 KB
  val_die.csv                        1,860.9 KB
  val_unit.csv                         313.4 KB
